# Análise celular
Depois cortar só coluna importante

In [19]:
import pandas as pd

## Entrada de Dados


In [20]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_endereco_dados = pd.read_csv(
    ENDERECO_DADOS,
    sep=';',
    encoding='iso-8859-1'
)

df_endereco_dados.columns = df_endereco_dados.columns.str.strip()

## Escolher colunas e fazer arquivo

Depois eu salvo tudo em CSV.

## Separar maiores e menores com quartis

Eu usar Q1 e Q3. Valor grande recebe `mais`. Valor pequeno recebe `menos`. Assim eu comparar lugar e tempo.

In [21]:
colunas_selecionadas = [
    'furto_celular',
    'roubo_celular',
    'total_roubos',
    'total_furtos',
    'cisp',
    'mes',
    'ano',
    'munic',
    'regiao'
]

df_celular = df_endereco_dados[colunas_selecionadas].copy()

# Corrige a região independentemente da forma de codificação do acento
df_celular['regiao'] = df_celular['regiao'].apply(
    lambda regiao: 'Grande Niterói'
    if 'Grande Niter' in str(regiao)
    else regiao
)

df_celular.to_csv('dados_celular.csv', sep=';', encoding='utf-8-sig', index=False)

df_celular.head()

,furto_celular,roubo_celular,total_roubos,total_furtos,cisp,mes,ano,munic,regiao
0,37,32,100,169,1,1,2003,Rio de Janeiro,Capital
1,23,14,82,115,4,1,2003,Rio de Janeiro,Capital
2,47,34,123,216,5,1,2003,Rio de Janeiro,Capital
3,8,20,143,90,6,1,2003,Rio de Janeiro,Capital
4,1,1,55,35,7,1,2003,Rio de Janeiro,Capital


In [22]:
# Disponibiliza todas as colunas de contexto
df_base = df_celular.reset_index(drop=True)

# Quartis para furtos de celular
Q1_furto = df_base['furto_celular'].quantile(0.25)
Q3_furto = df_base['furto_celular'].quantile(0.75)
df_base['flag_furto'] = df_base['furto_celular'].apply(
    lambda valor: 'mais' if valor >= Q3_furto else ('menos' if valor <= Q1_furto else 'medio')
)

# Quartis para roubos de celular
Q1_roubo = df_base['roubo_celular'].quantile(0.25)
Q3_roubo = df_base['roubo_celular'].quantile(0.75)
df_base['flag_roubo'] = df_base['roubo_celular'].apply(
    lambda valor: 'mais' if valor >= Q3_roubo else ('menos' if valor <= Q1_roubo else 'medio')
)

# Tudo permanece em um único arquivo, com as duas classificações
df_base.to_csv('dados_celular.csv', sep=';', encoding='utf-8-sig', index=False)

print('Furtos - Q1:', Q1_furto, '| Q3:', Q3_furto)
print('Roubos  - Q1:', Q1_roubo, '| Q3:', Q3_roubo)
print(df_base[['cisp', 'regiao', 'munic', 'furto_celular', 'flag_furto', 'roubo_celular', 'flag_roubo']].head())

Furtos - Q1: 1.0 | Q3: 15.0
Roubos  - Q1: 0.0 | Q3: 12.0
   cisp   regiao           munic  furto_celular flag_furto  roubo_celular  \
0     1  Capital  Rio de Janeiro             37       mais             32   
1     4  Capital  Rio de Janeiro             23       mais             14   
2     5  Capital  Rio de Janeiro             47       mais             34   
3     6  Capital  Rio de Janeiro              8      medio             20   
4     7  Capital  Rio de Janeiro              1      menos              1   

  flag_roubo  
0       mais  
1       mais  
2       mais  
3       mais  
4      medio  
